In [90]:
# pip install requests
# !pip install psycopg2
import requests
import pandas as pd
import json
import time
from datetime import datetime, timedelta
import psycopg2

In [91]:
# connect dataframe with AWS RDS database

endpoint = "ta43-onboarding.c5kcsm8im4cz.ap-southeast-2.rds.amazonaws.com"
master_username = "postgres"
password = "TA43Onboarding"
database = "postgres"
# port = 5432

conn_db = psycopg2.connect(host=endpoint, dbname=database, user=master_username, password=password)
cur = conn_db.cursor()

print("Connect to postgres")

Connect to postgres


In [92]:
# check connection

cur.execute("SELECT NOW();")
print(cur.fetchone()[0])  # prints current timestamp

2025-08-09 16:16:00.294090+00:00


In [93]:
# City of Melbourne token
headers = {
    "key_com" : "8272f3e3cf855ca3006bd9d38e135f06a0714adb8519bf503af526af"
}

In [94]:
# # Real-time: On-street parking bay sensors

# # On-street parking bay sensors
# onstreet_pbs_url = "https://data.melbourne.vic.gov.au/api/explore/v2.1/catalog/datasets/on-street-parking-bay-sensors/records"

# headers = {
#     "key_com" : "8272f3e3cf855ca3006bd9d38e135f06a0714adb8519bf503af526af"
# }
# # record limit per iteration
# record_limit = 100


# # loop interval: 2 min.
# loop_interval = 10

# # initialise current time - buffer
# current_time_buffer = (datetime.now() - timedelta(seconds=20)).isoformat()


# while True:
#     # initialise list for storing updated_data
#     updated_lst = []
#     offset = 0
#     while True:
#         condition = {"limit" : record_limit,
#                     "offset" : offset,
#                     "where" : f"lastupdated > '{current_time_buffer}'"}
        
#         # Get request
#         onstreet_pbs_response = requests.get(onstreet_pbs_url,
#                                             headers=headers,
#                                             params=condition)
#         # print test
#         test = onstreet_pbs_response.json()
#         print(test)

#         # HTTP breakout condition
#         res_status_cd = onstreet_pbs_response.status_code
#         if res_status_cd != 200:
#             if res_status_cd >= 500:
#                 print(f"Failed: {res_status_cd} Server Error")
#             else:
#                 print(f"Failed: {res_status_cd} Request Error", onstreet_pbs_response.text)
#             break

#         # Get response into record, csv format
#         record_json = onstreet_pbs_response.json()
#         record_onstreet_pbs = record_json.get("results", [])
#         print(f"Offset {offset}, fetched {len(record_onstreet_pbs)}")

#         # break condition if no record 
#         if not record_onstreet_pbs:
#             break
        
#         # append new record into list
#         updated_lst.extend(record_onstreet_pbs)

#         # update offest with record_limit
#         offset += record_limit

#     if updated_lst:
#         # convert json into pandas dataframe for data mapping
#         df_onstreet_pbs = pd.DataFrame(updated_lst)
#         print(f"{len(df_onstreet_pbs)} records updated.")

#         # update current time
#         current_time_buffer = max(pd.to_datetime(df_onstreet_pbs["lastupdated"]))
    
#     else:
#         print("No record updates.")
    
#     # loop report
#     time.sleep(loop_interval)

# print(df_onstreet_pbs)

In [95]:
# 1 (Get 100 records) On-street parking bay sensors
onstreet_pbs_url = "https://data.melbourne.vic.gov.au/api/explore/v2.1/catalog/datasets/on-street-parking-bay-sensors/records?limit=100"

# Get request
onstreet_pbs_response = requests.get(onstreet_pbs_url, headers=headers)

# Output
onstreet_pbs_data = onstreet_pbs_response.json()

In [120]:
# extract record list
record_onstreet_pbs = onstreet_pbs_data["results"]

# convert json to dataframe
df_onstreet_pbs = pd.json_normalize(record_onstreet_pbs)

# rename column header
df_onstreet_pbs = df_onstreet_pbs.rename(
    columns={"zone_number" : "parkingzone",
             "status_description" : "status_desc", 
             "location.lon" : "longitude",
             "location.lat" : "latitude"}
    )

# convert datetime columns
df_onstreet_pbs["lastupdated"] = pd.to_datetime(df_onstreet_pbs["lastupdated"])
# df_onstreet_pbs["lastupdated"] = df_onstreet_pbs["lastupdated"].dt.strftime("%H:%M:%S")

df_onstreet_pbs["status_timestamp"] = pd.to_datetime(df_onstreet_pbs["status_timestamp"])
# df_onstreet_pbs["status_timestamp"] = df_onstreet_pbs["status_timestamp"].dt.strftime("%H:%M:%S")
df_onstreet_pbs["parking_date"] = df_onstreet_pbs["status_timestamp"].dt.date
df_onstreet_pbs["parking_time"] = df_onstreet_pbs["status_timestamp"].dt.time

df_onstreet_pbs["parkingzone"] = df_onstreet_pbs["parkingzone"].astype("Int64")


def parking_status(x):
    """
    Function convert parking status.
    Parameter: x is parking bay sensor record from status_description
    Returns:
    1 if parking is available (Unoccupied),
    0 if parking is not available (Present)
    """
    if x["status_desc"] == "Unoccupied":
        return True
    else:
        return False

df_onstreet_pbs["is_available"] = df_onstreet_pbs.apply(parking_status, axis=1)

# output
# print(df_onstreet_pbs.head())
# print(df_onstreet_pbs.dtypes)


# For SQL create table: PARKING_BAY_SENSOR (pk = kerbsideid, fk = parkingzone)
tb_parking_bay_sensor = df_onstreet_pbs[["kerbsideid", "lastupdated", "parking_date", "parking_time", "is_available", "parkingzone"]]

In [97]:
# # 2. Sign plates located in each parking zone
# sign_plates_loc_url = "https://data.melbourne.vic.gov.au/api/explore/v2.1/catalog/datasets/sign-plates-located-in-each-parking-zone/records?limit=100"

# # Get request
# sign_plates_loc_response = requests.get(sign_plates_loc_url, headers=headers)

# # Output
# sign_plates_loc_data = sign_plates_loc_response.json()

In [98]:
# # extract record list
# record_sign_plates_loc = sign_plates_loc_data["results"]

# # convert json to dataframe
# df_sign_plates_loc = pd.json_normalize(record_sign_plates_loc)

# # convert datetime columns
# df_sign_plates_loc["time_restrictions_start"] = pd.to_datetime(df_sign_plates_loc["time_restrictions_start"])
# df_sign_plates_loc["time_restrictions_start"] = df_sign_plates_loc["time_restrictions_start"].dt.time

# df_sign_plates_loc["time_restrictions_finish"] = pd.to_datetime(df_sign_plates_loc["time_restrictions_finish"])
# df_sign_plates_loc["time_restrictions_finish"] = df_sign_plates_loc["time_restrictions_finish"].dt.time

# # for checking
# print(df_sign_plates_loc)

# print(df_sign_plates_loc["restriction_display"].unique())

# # # Day of week dict {day:idx}
# # day_dict = {
# #     ""
# # }

In [139]:
# 2. (Static record) Sign plates located in each parking zone
with open("Dataset/sign-plates-located-in-each-parking-zone.json", "r") as file:
    data_sign_plates = json.load(file)

# convert json to dataframe
df_sign_plates = pd.json_normalize(data_sign_plates)

# format column data type
df_sign_plates["time_restrictions_start"] = pd.to_datetime(df_sign_plates["time_restrictions_start"], format="%H:%M:%S")
df_sign_plates["time_restrictions_start"] = df_sign_plates["time_restrictions_start"].dt.time

df_sign_plates["time_restrictions_finish"] = pd.to_datetime(df_sign_plates["time_restrictions_finish"], format="%H:%M:%S")
df_sign_plates["time_restrictions_finish"] = df_sign_plates["time_restrictions_finish"].dt.time
# output
# print(df_sign_plates.head())
# print(df_sign_plates.dtypes)

# For SQL create table: SIGN_PLATE (pk = (parkingzone, restruction_days), fk = parkingzone)
tb_sign_plates = (df_sign_plates
                  .dropna(subset=["parkingzone","restriction_days"])
                  .assign(parkingzone=pd.to_numeric(df_sign_plates["parkingzone"], errors="coerce"))
                  .astype({"parkingzone":"int64"})
                  .drop_duplicates(subset=["parkingzone","restriction_days"]))

In [100]:
# count_dist= df_sign_plates["parkingzone"].unique()
# pd.value_counts(count_dist)

In [101]:
# # 3. Parking zone linked to street segments
# parking_zone_url = "https://data.melbourne.vic.gov.au/api/explore/v2.1/catalog/datasets/parking-zones-linked-to-street-segments/records?limit=100"

# # Get request
# parking_zone_response = requests.get(parking_zone_url, headers=headers)

# # Output
# parking_zone_data = parking_zone_response.json()

In [102]:
# # extract record list
# record_parking_zone = parking_zone_data["results"]

# # convert json to dataframe
# record_parking_zone = pd.json_normalize(record_parking_zone)

# print(record_parking_zone)
# print(record_parking_zone.dtypes)

In [ ]:
# 3. (Static record) Parking zone linked to street segments
with open("Dataset/parking-zones-linked-to-street-segments.json", "r") as file:
    data_parking_zones = json.load(file)

df_parking_zones = pd.json_normalize(data_parking_zones)
print(df_parking_zones)

zones_all = pd.concat(
    [
        df_parking_zones["parkingzone"],   # segments
        df_sign_plates["parkingzone"],     # sign plates
        df_onstreet_pbs["parkingzone"],    # sensors (real-time)
    ],
    ignore_index=True
)

# For SQL create table: PARKING_ZONE (pk = parkingzone)
tb_parking_zones = pd.to_numeric(df_parking_zones["parkingzone"]).dropna().drop_duplicates().to_frame("parkingzone")

     parkingzone         onstreet            streetfrom            streetto  \
0           7000      Poplar Road       Upfield Railway      Kendall Avenue   
1           7031  Cardigan Street    Argyle Place North      Grattan Street   
2           7068     Lygon Street        Faraday Street        Elgin Street   
3           7067     Lygon Street         Pelham Street  Argyle Place North   
4           7118  Swanston Street  Lincoln Square North      Grattan Street   
..           ...              ...                   ...                 ...   
793         7963   Rosslyn Street         Howard Street         King Street   
794         7950    Railway Place        Stanley Street        Roden Street   
795         7968   Spencer Street       La Trobe Street     Jeffcott Street   
796         7975   Stanley Street           King Street      Spencer Street   
797         7995   William Street          Capel Street      Rosslyn Street   

     segment_id  
0         22405  
1         20512

In [138]:
# 4. (Static record) Parking zone linked to street segments

# For SQL create table: PARKING_ZONE_SEGMENT (pk = (parkingzone,segment_id), fk = parkingzone,segment_id)
tb_parking_zone_segment = (df_parking_zones[["parkingzone", "segment_id"]]
                           .assign(
                               parkingzone=pd.to_numeric(df_parking_zones["parkingzone"], errors="coerce"),
                               segment_id=pd.to_numeric(df_parking_zones["segment_id"], errors="coerce"))
                               .dropna(subset=["parkingzone", "segment_id"])
                               .astype({"parkingzone":"int64","segment_id":"int64"})
                               .drop_duplicates(subset=["parkingzone","segment_id"]))

In [ ]:
# 5. (Static record) Parking zone linked to street segments

# For SQL create table: PARKING_SEGMENT (pk = segment_id)
tb_parking_segment = (df_parking_zones[["segment_id", "onstreet", "streetfrom", "streetto"]]
                      .dropna(subset=["segment_id"])
                      .drop_duplicates(subset=["segment_id"]))

In [106]:
# 6. (static) On-street parking bays
with open("Dataset/on-street-parking-bays.json", "r") as file:
    data_parking_bays = json.load(file)

# convert json to dataframe
df_parking_bays = pd.json_normalize(data_parking_bays)

# rename column
df_parking_bays = df_parking_bays.rename(columns={"roadsegmentid" : "segment_id",
                                                  "roadsegmentdescription" : "segment_desc"})

# format column data type
df_parking_bays["kerbsideid"] = pd.to_numeric(df_parking_bays["kerbsideid"], errors="coerce")

df_parking_bays["lastupdated"] = pd.to_datetime(df_parking_bays["lastupdated"], format="%Y-%m-%d")

# filter necessary column
df_parking_bays = df_parking_bays.drop(columns=["location.lon","location.lat"])


# print(df_parking_bays.head())
# print(df_parking_bays.dtypes)

# For SQL create table: PARKING_BAY (pk = kerbsideid, fk = segment_id)
tb_parking_bays = df_parking_bays[["kerbsideid", "segment_id", "segment_desc", "latitude", "longitude"]]

In [ ]:
# create engine for connection
from sqlalchemy import create_engine, text

endpoint = "ta43-onboarding.c5kcsm8im4cz.ap-southeast-2.rds.amazonaws.com"
database = "postgres"  # or your actual DB
username = "postgres"
password = "TA43Onboarding"

rds_engine = create_engine(
    f"postgresql+psycopg2://{username}:{password}@{endpoint}:5432/{database}?sslmode=require"
)

# check connection
# with rds_engine.connect() as conn:
#     print(conn.execute(text("SELECT now()")).scalar())

2025-08-09 16:16:01.803574+00:00


In [154]:
from sqlalchemy import text
from uuid import uuid4
import re

SCHEMA = "public"   # change if your tables live in a different schema

def _qident(name: str) -> str:
    """Safely quote a SQL identifier (very basic validation)."""
    if not re.match(r'^[A-Za-z_][A-Za-z0-9_]*$', name):
        raise ValueError(f"Unsafe identifier: {name}")
    return f'"{name}"'

def upsert_do_nothing(df, target_table, pk_cols, conn, schema=SCHEMA, chunksize=10_000, cast_map=None):
    # Reflect target table columns (ordered)
    cols_rs = conn.execute(text("""
        SELECT column_name
        FROM information_schema.columns
        WHERE table_schema = :schema AND table_name = :table
        ORDER BY ordinal_position
    """), {"schema": schema, "table": target_table})
    target_cols = [r[0] for r in cols_rs]

    # Ensure PK columns exist on target
    missing_pks = [c for c in pk_cols if c not in target_cols]
    if missing_pks:
        raise ValueError(f"PK columns not found in {target_table}: {missing_pks}")

    # Only load columns that exist on target (keeps order)
    cols_to_load = [c for c in target_cols if c in df.columns]
    if not cols_to_load:
        raise ValueError(f"No overlapping columns between DataFrame and {target_table}.")

    # Unique staging table name
    stage_table = f"_stg_{target_table}_{uuid4().hex[:8]}"

    # Write DF to staging
    df.to_sql(
        stage_table,
        conn,
        schema=schema,
        if_exists="replace",
        index=False,
        method="multi",
        chunksize=chunksize,
    )

    # Build and run UPSERT
    col_list = ", ".join(_qident(c) for c in cols_to_load)
    pk_list  = ", ".join(_qident(c) for c in pk_cols)
    full_target = f'{_qident(schema)}.{_qident(target_table)}'
    full_stage  = f'{_qident(schema)}.{_qident(stage_table)}'

    sel_exprs = []
    for c in cols_to_load:
        if cast_map and c in cast_map:
            sel_exprs.append(f'{_qident(c)}::{cast_map[c]}')
        else:
            sel_exprs.append(_qident(c))
    sel_list = ", ".join(sel_exprs)

    conn.execute(text(f"""
        INSERT INTO {full_target} ({col_list})
        SELECT {sel_list}
        FROM {full_stage}
        ON CONFLICT ({pk_list}) DO NOTHING
    """))

    # Drop staging
    conn.execute(text(f"DROP TABLE {full_stage}"))

def to_str_set(s: pd.Series) -> set:
    # drop NA/NaN and normalize to strings so set math and sorting are safe
    return set(s.dropna().astype(str))

# ----------------- Use it for every table (parents first) -----------------

# parents first
with rds_engine.begin() as conn:
    # 1) PARKING_ZONE (pk = parkingzone)
    upsert_do_nothing(tb_parking_zones, "parking_zone", ["parkingzone"], conn)

    # 2) PARKING_SEGMENT (pk = segment_id)
    upsert_do_nothing(tb_parking_segment, "parking_segment", ["segment_id"], conn)

    # ---- fetch current parent keys from DB
    zones_db = to_str_set(pd.read_sql_query('SELECT parkingzone FROM parking_zone', conn)['parkingzone'])
    segs_db  = to_str_set(pd.read_sql_query('SELECT segment_id FROM parking_segment', conn)['segment_id'])

    # ---- filter children by existing parents + small diagnostics
    # 3) SIGN_PLATE (fk: parkingzone)
    sp_missing = sorted(list(to_str_set(tb_sign_plates['parkingzone']) - zones_db))
    sp_ok = tb_sign_plates[tb_sign_plates['parkingzone'].astype(str).isin(zones_db)]

    # 4) PARKING_ZONE_SEGMENT (fk: parkingzone, segment_id)
    pzs_missing_zone = sorted(list(to_str_set(tb_parking_zone_segment['parkingzone']) - zones_db))
    pzs_missing_seg  = sorted(list(to_str_set(tb_parking_zone_segment['segment_id'])  - segs_db))
    pzs_ok = tb_parking_zone_segment[
        tb_parking_zone_segment['parkingzone'].astype(str).isin(zones_db) &
        tb_parking_zone_segment['segment_id'].astype(str).isin(segs_db)
    ]

    # 5) PARKING_BAY (fk: segment_id)
    bays_missing = sorted(list(to_str_set(tb_parking_bays['segment_id']) - segs_db))
    bays_ok = tb_parking_bays[tb_parking_bays['segment_id'].astype(str).isin(segs_db)]


    # 6) PARKING_BAY_SENSOR (fk: parkingzone)
    sens_missing = to_str_set(tb_parking_bay_sensor['parkingzone']) - zones_db
    sens_ok = tb_parking_bay_sensor[
        tb_parking_bay_sensor['parkingzone'].notna() &
        tb_parking_bay_sensor['parkingzone'].astype(str).isin(zones_db)
    ]

    # --- when building filtered frames
    sp_ok  = tb_sign_plates[tb_sign_plates['parkingzone'].astype(str).isin(zones_db)].copy()
    pzs_ok = tb_parking_zone_segment[
        tb_parking_zone_segment['parkingzone'].astype(str).isin(zones_db) &
        tb_parking_zone_segment['segment_id'].astype(str).isin(segs_db)
    ].copy()
    bays_ok = tb_parking_bays[tb_parking_bays['segment_id'].astype(str).isin(segs_db)].copy()
    bays_ok = bays_ok[bays_ok['kerbsideid'].notna()]     # <<< drop null PKs
    sens_ok = tb_parking_bay_sensor[
        tb_parking_bay_sensor['parkingzone'].notna() &
        tb_parking_bay_sensor['parkingzone'].astype(str).isin(zones_db)
    ].copy()

    # avoid SettingWithCopyWarning for time cleanup
    for c in ["time_restrictions_start", "time_restrictions_finish"]:
        sp_ok.loc[:, c] = sp_ok[c].astype(str).str.strip().replace({"": None, "NaT": None, "nan": None})


    # ---- now upsert the filtered children
    upsert_do_nothing(sp_ok, "sign_plate", ["parkingzone", "restriction_days"], conn, cast_map={"time_restrictions_start": "time", "time_restrictions_finish": "time",})
    upsert_do_nothing(pzs_ok, "parking_zone_segment", ["parkingzone", "segment_id"], conn)
    upsert_do_nothing(bays_ok, "parking_bay", ["kerbsideid"], conn)
    upsert_do_nothing(sens_ok, "parking_bay_sensor", ["kerbsideid"], conn)


In [ ]:
# export dataframe from vs code to aws rds postgresql


# STATIC DB
# 1. For SQL create table: PARKING_ZONE (pk = parkingzone)
# tb_parking_zones.to_sql("parking_zone", rds_engine, if_exists="append", index=False, chunksize=10_000, method="multi")
# Insert only zones not already in DB
existing = pd.read_sql_query("SELECT parkingzone FROM parking_zone", rds_engine)
new_zones = tb_parking_zones[~tb_parking_zones["parkingzone"].isin(existing["parkingzone"])]
if not new_zones.empty:
    new_zones.to_sql("parking_zone", rds_engine, if_exists="append", index=False, method="multi")


# 2.  For SQL create table: PARKING_SEGMENT (pk = segment_id)
tb_parking_segment.to_sql("parking_segment", rds_engine, if_exists="append", index=False, chunksize=10_000, method="multi")

# 3. For SQL create table: SIGN_PLATE (pk = (parkingzone, restruction_days), fk = parkingzone)
tb_sign_plates.to_sql("sign_plate", rds_engine, if_exists="append", index=False, chunksize=10_000, method="multi")

# 4. For SQL create table: PARKING_ZONE_SEGMENT (pk = (parkingzone,segment_id), fk = parkingzone,segment_id)
tb_parking_zone_segment.to_sql("parking_zone_segment", rds_engine, if_exists="append", index=False, chunksize=10_000, method="multi")

# 5. For SQL create table: PARKING_BAY (pk = kerbsideid, fk = segment_id)
tb_parking_bays.to_sql("parking_bay", rds_engine, if_exists="append", index=False, chunksize=10_000, method="multi")

# REAL-TIME DB
# 6. create table: PARKING_BAY_SENSOR (pk = kerbsideid, fk = parkingzone)
tb_parking_bay_sensor.to_sql("parking_bay_sensor", rds_engine, if_exists="append", index=False, chunksize=10_000, method="multi")


IntegrityError: (psycopg2.errors.UniqueViolation) duplicate key value violates unique constraint "parking_segment_pkey"
DETAIL:  Key (segment_id)=(22405) already exists.

[SQL: INSERT INTO parking_segment (segment_id, onstreet, streetfrom, streetto) VALUES (%(segment_id_m0)s, %(onstreet_m0)s, %(streetfrom_m0)s, %(streetto_m0)s), (%(segment_id_m1)s, %(onstreet_m1)s, %(streetfrom_m1)s, %(streetto_m1)s), (%(segment_id_m2)s, %(onstreet_m2)s, %(streetfrom_m2)s, %(streetto_m2)s), (%(segment_id_m3)s, %(onstreet_m3)s, %(streetfrom_m3)s, %(streetto_m3)s), (%(segment_id_m4)s, %(onstreet_m4)s, %(streetfrom_m4)s, %(streetto_m4)s), (%(segment_id_m5)s, %(onstreet_m5)s, %(streetfrom_m5)s, %(streetto_m5)s), (%(segment_id_m6)s, %(onstreet_m6)s, %(streetfrom_m6)s, %(streetto_m6)s), (%(segment_id_m7)s, %(onstreet_m7)s, %(streetfrom_m7)s, %(streetto_m7)s), (%(segment_id_m8)s, %(onstreet_m8)s, %(streetfrom_m8)s, %(streetto_m8)s), (%(segment_id_m9)s, %(onstreet_m9)s, %(streetfrom_m9)s, %(streetto_m9)s), (%(

In [116]:
tb_parking_bay_sensor.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype              
---  ------        --------------  -----              
 0   kerbsideid    100 non-null    int64              
 1   lastupdated   100 non-null    datetime64[ns, UTC]
 2   parking_date  100 non-null    object             
 3   parking_time  100 non-null    object             
 4   is_available  100 non-null    int64              
 5   parkingzone   90 non-null     Int64              
dtypes: Int64(1), datetime64[ns, UTC](1), int64(2), object(2)
memory usage: 4.9+ KB
